# Lab 2: Phát hiện ảnh GAN giả mạo bằng CNN (Colab GPU)

MSHV 25C15019 — Toán cho AI, HCMUS.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nmnhut-it/math-for-ai/blob/master/lab2_cnn/colab.ipynb)  
Source: <https://github.com/nmnhut-it/math-for-ai/tree/master/lab2_cnn>

**Self-contained**: không cần git clone. Code của từng thí nghiệm được ghi thẳng vào notebook qua `%%writefile`, chạy xong là cell kế tiếp đọc được.

**Trước khi chạy**: Runtime > Change runtime type > **GPU** (T4 free hoặc L4).

Pipeline `Run all` (~20-25 phút trên L4):

1. cGAN-MNIST in-house (MLP) + TinyCNN scratch (~99%) và Grad-CAM
2. PGAN-DTD + TexCNN scratch (~62%, baseline)
3. PGAN-DTD + ResNet18 transfer (~99%)
4. BigGAN-128 + Imagenette + ResNet18 transfer (~99%)
5. Grad-CAM ResNet18 cho PGAN và BigGAN
6. Cross-test: ResNet18(BigGAN) áp lên PGAN để kiểm tra tính tổng quát


## 1. Setup runtime

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('!!! Khong co GPU - bat GPU runtime truoc khi chay !!!')


In [ ]:
!pip install -q pytorch-pretrained-biggan

In [ ]:
# Tạo thư mục làm việc và data sibling theo đúng cấu trúc khi chạy local.
import os
os.makedirs('/content/lab2_cnn', exist_ok=True)
os.makedirs('/content/data', exist_ok=True)
os.chdir('/content/lab2_cnn')
os.makedirs('output', exist_ok=True)
print('cwd:', os.getcwd())


## 2. Ghi source code ra đĩa

Mỗi cell dưới đây dùng `%%writefile` để xuất một script trong `src/` thành file Python tại `/content/lab2_cnn/`. Chưa chạy gì.

In [ ]:
%%writefile exp1_cgan_tinycnn.py
# TinyCNN phan biet MNIST that vs cGAN fake
# Self-contained: inline cGAN (Mirza & Osindero 2014) + tu train neu chua co checkpoint.
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR     = "output"
DATA_DIR    = "../data"
CGAN_CKPTS  = ["output/cG_final.pth", "../lab2/output/cG_final.pth"]
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
N_PER_CLASS = 10000
BATCH       = 64
EPOCHS      = 5
LR          = 1e-3
SEED        = 42
VAL_RATIO   = 0.2

Z_DIM       = 100
NUM_CLASSES = 10
EMBED_DIM   = 10
IMG_SIZE    = 28
IMG_DIM     = IMG_SIZE * IMG_SIZE
GAN_EPOCHS  = 30
GAN_BATCH   = 256
LR_GAN      = 2e-4
BETA1       = 0.5

os.makedirs(OUT_DIR, exist_ok=True)


def _mlp_block(in_f, out_f, dropout=0.0, bn=False):
    layers = [nn.Linear(in_f, out_f)]
    if bn:
        layers.append(nn.BatchNorm1d(out_f))
    layers.append(nn.LeakyReLU(0.2, inplace=True))
    if dropout > 0:
        layers.append(nn.Dropout(dropout))
    return layers


class ConditionalGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(NUM_CLASSES, EMBED_DIM)
        self.net = nn.Sequential(
            *_mlp_block(Z_DIM + EMBED_DIM, 256, bn=True),
            *_mlp_block(256, 512, bn=True),
            *_mlp_block(512, 1024, bn=True),
            nn.Linear(1024, IMG_DIM), nn.Tanh(),
        )

    def forward(self, z, y):
        h = torch.cat([z, self.label_emb(y)], dim=1)
        return self.net(h).view(-1, 1, IMG_SIZE, IMG_SIZE)


class ConditionalDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(NUM_CLASSES, EMBED_DIM)
        self.net = nn.Sequential(
            *_mlp_block(IMG_DIM + EMBED_DIM, 1024, dropout=0.3),
            *_mlp_block(1024, 512, dropout=0.3),
            *_mlp_block(512, 256, dropout=0.3),
            nn.Linear(256, 1), nn.Sigmoid(),
        )

    def forward(self, x, y):
        h = torch.cat([x.view(-1, IMG_DIM), self.label_emb(y)], dim=1)
        return self.net(h)


def train_cgan(log_fn):
    log_fn("\n" + "=" * 60); log_fn("Train cGAN tu dau (chua co checkpoint)"); log_fn("=" * 60)
    tf = transforms.Compose([transforms.ToTensor(),
                             transforms.Normalize([0.5], [0.5])])
    mnist = datasets.MNIST(DATA_DIR, train=True, download=True, transform=tf)
    loader = DataLoader(mnist, batch_size=GAN_BATCH, shuffle=True, drop_last=True)

    G = ConditionalGenerator().to(DEVICE)
    D = ConditionalDiscriminator().to(DEVICE)
    opt_G = optim.Adam(G.parameters(), lr=LR_GAN, betas=(BETA1, 0.999))
    opt_D = optim.Adam(D.parameters(), lr=LR_GAN, betas=(BETA1, 0.999))
    bce = nn.BCELoss()

    for epoch in range(1, GAN_EPOCHS + 1):
        for real, labels in loader:
            real, labels = real.to(DEVICE), labels.to(DEVICE)
            bs = real.size(0)
            ones  = torch.ones (bs, 1, device=DEVICE)
            zeros = torch.zeros(bs, 1, device=DEVICE)
            with torch.no_grad():
                z = torch.randn(bs, Z_DIM, device=DEVICE)
                fake = G(z, labels)
            loss_D = (bce(D(real, labels), ones) + bce(D(fake, labels), zeros)) / 2
            opt_D.zero_grad(); loss_D.backward(); opt_D.step()
            y_g = torch.randint(0, NUM_CLASSES, (bs,), device=DEVICE)
            z2 = torch.randn(bs, Z_DIM, device=DEVICE)
            fake2 = G(z2, y_g)
            loss_G = bce(D(fake2, y_g), ones)
            opt_G.zero_grad(); loss_G.backward(); opt_G.step()
        log_fn(f"  Epoch {epoch:3d}  loss_D={loss_D.item():.3f}  loss_G={loss_G.item():.3f}")

    ckpt = f"{OUT_DIR}/cG_final.pth"
    torch.save(G.state_dict(), ckpt)
    log_fn(f"  Saved {ckpt}")
    return G


def load_or_train_cgan(log_fn):
    for ckpt in CGAN_CKPTS:
        if os.path.exists(ckpt):
            G = ConditionalGenerator().to(DEVICE)
            G.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
            log_fn(f"  Loaded cGAN checkpoint: {ckpt}")
            return G
    return train_cgan(log_fn)


def build_dataset(log_fn=print):
    log_fn("\n" + "=" * 60); log_fn("Build dataset"); log_fn("=" * 60)

    tf = transforms.Compose([transforms.ToTensor(),
                             transforms.Normalize([0.5], [0.5])])
    mnist = datasets.MNIST(DATA_DIR, train=True, download=True, transform=tf)
    loader = DataLoader(mnist, batch_size=N_PER_CLASS, shuffle=True)
    reals, _ = next(iter(loader))
    log_fn(f"  Reals: {reals.shape}  range [{reals.min():.2f}, {reals.max():.2f}]")

    G = load_or_train_cgan(log_fn)
    G.train(False)
    z = torch.randn(N_PER_CLASS, Z_DIM, device=DEVICE)
    y_g = torch.randint(0, NUM_CLASSES, (N_PER_CLASS,), device=DEVICE)
    with torch.no_grad():
        fakes = G(z, y_g).cpu()
    log_fn(f"  Fakes: {fakes.shape}  range [{fakes.min():.2f}, {fakes.max():.2f}]")

    X = torch.cat([reals, fakes], dim=0)
    y = torch.cat([torch.zeros(N_PER_CLASS, dtype=torch.long),
                   torch.ones (N_PER_CLASS, dtype=torch.long)])
    log_fn(f"  Combined: X={X.shape}  y={y.shape}")
    log_fn(f"  Label convention: 0 = real, 1 = fake")
    return X, y


class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,  16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(32 * 7 * 7, 64)
        self.fc2   = nn.Linear(64, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_one_epoch(model, loader, opt, loss_fn):
    model.train()
    total_loss = 0.0; correct = 0; n = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        logits = model(X)
        loss = loss_fn(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item() * X.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += X.size(0)
    return total_loss / n, correct / n


def evaluate(model, loader, loss_fn):
    model.train(False)
    total_loss = 0.0; correct = 0; n = 0
    all_pred = []; all_true = []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            logits = model(X)
            loss = loss_fn(logits, y)
            total_loss += loss.item() * X.size(0)
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            n += X.size(0)
            all_pred.append(pred.cpu()); all_true.append(y.cpu())
    return (total_loss / n, correct / n,
            torch.cat(all_pred).numpy(), torch.cat(all_true).numpy())


def confusion_matrix(y_true, y_pred):
    cm = np.zeros((2, 2), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def plot_confusion(cm, fname):
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["real (pred)", "fake (pred)"])
    ax.set_yticklabels(["real (true)", "fake (true)"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black",
                    fontsize=14, fontweight="bold")
    ax.set_title("Confusion matrix (validation set)")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/{fname}", dpi=130); plt.close()


def main():
    LOG = open(f"{OUT_DIR}/results.txt", "w", encoding="utf-8")
    def log(m=""):
        print(m); LOG.write(m + "\n"); LOG.flush()

    torch.manual_seed(SEED); np.random.seed(SEED)
    log(f"Device: {DEVICE}")
    log(f"N_PER_CLASS={N_PER_CLASS}  EPOCHS={EPOCHS}  BATCH={BATCH}  LR={LR}")

    X, y = build_dataset(log)
    ds = TensorDataset(X, y)
    n_val = int(len(ds) * VAL_RATIO)
    n_train = len(ds) - n_val
    train_ds, val_ds = random_split(ds, [n_train, n_val],
                                    generator=torch.Generator().manual_seed(SEED))
    log(f"  Train/Val split: {n_train}/{n_val}")

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False)

    log("\n" + "=" * 60); log("Build TinyCNN"); log("=" * 60)
    model = TinyCNN().to(DEVICE)
    log(f"  Parameters: {count_params(model):,}")
    log(f"  Architecture:")
    log(f"    conv1: (1->16, 3x3) -> ReLU -> MaxPool(2)  ->  (16,14,14)")
    log(f"    conv2: (16->32, 3x3) -> ReLU -> MaxPool(2) ->  (32, 7, 7)")
    log(f"    fc1:   (1568 -> 64) -> ReLU")
    log(f"    fc2:   (64 -> 2) logits")

    opt = optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()

    log("\n" + "=" * 60); log("Training"); log("=" * 60)
    log(f"  {'Epoch':>6} {'TrainLoss':>10} {'TrainAcc':>10} {'ValLoss':>10} {'ValAcc':>10}")
    best_val_acc = 0.0
    for epoch in range(1, EPOCHS + 1):
        tl, ta = train_one_epoch(model, train_loader, opt, loss_fn)
        vl, va, _, _ = evaluate(model, val_loader, loss_fn)
        log(f"  {epoch:>6d} {tl:>10.4f} {ta:>10.4f} {vl:>10.4f} {va:>10.4f}")
        if va > best_val_acc:
            best_val_acc = va
            torch.save(model.state_dict(), f"{OUT_DIR}/cnn_best.pth")
    log(f"\n  Best val acc: {best_val_acc:.4f}  (saved to {OUT_DIR}/cnn_best.pth)")

    log("\n" + "=" * 60); log("Final evaluation (best checkpoint)"); log("=" * 60)
    model.load_state_dict(torch.load(f"{OUT_DIR}/cnn_best.pth",
                                      map_location=DEVICE, weights_only=True))
    vl, va, pred, true = evaluate(model, val_loader, loss_fn)
    log(f"  Val accuracy: {va:.4f}")
    log(f"  Val loss:     {vl:.4f}")

    cm = confusion_matrix(true, pred)
    log(f"\n  Confusion matrix:")
    log(f"                 pred=real  pred=fake")
    log(f"    true=real    {cm[0,0]:>9d}  {cm[0,1]:>9d}")
    log(f"    true=fake    {cm[1,0]:>9d}  {cm[1,1]:>9d}")
    log(f"\n    Real recall: {cm[0,0]/(cm[0,0]+cm[0,1]):.4f}")
    log(f"    Fake recall: {cm[1,1]/(cm[1,0]+cm[1,1]):.4f}")
    plot_confusion(cm, "confusion_matrix.png")
    log(f"  Saved {OUT_DIR}/confusion_matrix.png")

    torch.save({
        "X": X, "y": y,
        "val_indices": val_ds.indices,
    }, f"{OUT_DIR}/dataset.pt")
    log(f"  Saved {OUT_DIR}/dataset.pt for Grad-CAM script")

    LOG.close()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile gradcam_tinycnn.py
# Grad-CAM cho TinyCNN, kèm tương quan với high-freq residual để xem CNN có thật
# bám vào vùng pixel jitter (dấu vân tay đặc trưng của cGAN MLP) hay không.
import os, sys
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

sys.path.append(os.path.dirname(os.path.abspath(__file__)))
from exp1_cgan_tinycnn import TinyCNN

OUT_DIR  = "output"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
N_VIS    = 4
SEED     = 7

torch.manual_seed(SEED); np.random.seed(SEED)


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients   = None
        target_layer.register_forward_hook(self._fwd_hook)
        target_layer.register_full_backward_hook(self._bwd_hook)

    def _fwd_hook(self, module, inp, out):
        self.activations = out.detach()

    def _bwd_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, x, class_idx):
        self.model.zero_grad()
        logits = self.model(x)
        score  = logits[:, class_idx].sum()
        score.backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=x.shape[2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze(1)
        for i in range(cam.size(0)):
            mn, mx = cam[i].min(), cam[i].max()
            if mx - mn > 1e-8:
                cam[i] = (cam[i] - mn) / (mx - mn)
            else:
                cam[i] = torch.zeros_like(cam[i])
        return cam.cpu().numpy()


def high_freq_residual(x, kernel_size=3):
    # |I - blur(I)| đo lượng năng lượng tần số cao tại từng pixel; pixel jitter của
    # cGAN-MLP biểu hiện thành các đốm residual lớn.
    blur = F.avg_pool2d(x, kernel_size=kernel_size, stride=1,
                        padding=kernel_size // 2)
    res = (x - blur).abs().squeeze(1)
    out = torch.zeros_like(res)
    for i in range(res.size(0)):
        mn, mx = res[i].min(), res[i].max()
        if mx - mn > 1e-8:
            out[i] = (res[i] - mn) / (mx - mn)
    return out.cpu().numpy()


print("Loading model + dataset...")
model = TinyCNN().to(DEVICE)
model.load_state_dict(torch.load(f"{OUT_DIR}/cnn_best.pth", map_location=DEVICE,
                                  weights_only=True))
model.train(False)

cache = torch.load(f"{OUT_DIR}/dataset.pt", weights_only=True)
X = cache["X"]; y = cache["y"]; val_idx = cache["val_indices"]

val_idx = np.array(val_idx)
val_y = y[val_idx].numpy()
real_pool = val_idx[val_y == 0]
fake_pool = val_idx[val_y == 1]

rng = np.random.RandomState(SEED)
real_sel = rng.choice(real_pool, size=N_VIS, replace=False)
fake_sel = rng.choice(fake_pool, size=N_VIS, replace=False)

X_real = X[real_sel].to(DEVICE)
X_fake = X[fake_sel].to(DEVICE)
print(f"  Selected {N_VIS} real + {N_VIS} fake from val set")


gradcam = GradCAM(model, target_layer=model.conv2)
cam_real = gradcam(X_real.clone().requires_grad_(True), class_idx=1)
cam_fake = gradcam(X_fake.clone().requires_grad_(True), class_idx=1)

res_real = high_freq_residual(X_real)
res_fake = high_freq_residual(X_fake)

with torch.no_grad():
    logits_real = model(X_real).cpu().numpy()
    logits_fake = model(X_fake).cpu().numpy()
prob_real = np.exp(logits_real) / np.exp(logits_real).sum(axis=1, keepdims=True)
prob_fake = np.exp(logits_fake) / np.exp(logits_fake).sum(axis=1, keepdims=True)


def pearson_per_image(a, b):
    out = np.zeros(a.shape[0])
    for i in range(a.shape[0]):
        av = a[i].ravel(); bv = b[i].ravel()
        av = av - av.mean(); bv = bv - bv.mean()
        denom = np.sqrt((av * av).sum() * (bv * bv).sum())
        out[i] = (av * bv).sum() / denom if denom > 1e-8 else 0.0
    return out


# Tính trên 200 ảnh mỗi lớp để có Pearson r đáng tin (4 mẫu không đủ).
print("Computing Pearson r over full val set...")
N_CORR = 200
real_pool_sel = rng.choice(real_pool, size=N_CORR, replace=False)
fake_pool_sel = rng.choice(fake_pool, size=N_CORR, replace=False)
X_real_full = X[real_pool_sel].to(DEVICE)
X_fake_full = X[fake_pool_sel].to(DEVICE)
cam_real_full = gradcam(X_real_full.clone().requires_grad_(True), class_idx=1)
cam_fake_full = gradcam(X_fake_full.clone().requires_grad_(True), class_idx=1)
res_real_full = high_freq_residual(X_real_full)
res_fake_full = high_freq_residual(X_fake_full)
r_real = pearson_per_image(res_real_full, cam_real_full)
r_fake = pearson_per_image(res_fake_full, cam_fake_full)

# MNIST thật có nền hoàn toàn -1 (đen tuyệt đối); cGAN-MLP để lại jitter nhỏ
# khiến nền lệch khỏi -1. Đo trực tiếp bằng mean |x − (−1)| trên các pixel mà
# cả 3×3 lân cận đều rất tối (xa stroke chữ).
def pure_background_deviation(x, neighbor_thresh=-0.85):
    nbh_max = F.max_pool2d(x, kernel_size=3, stride=1, padding=1)
    pure_mask = (nbh_max < neighbor_thresh).squeeze(1).float()
    deviation = (x.squeeze(1) - (-1.0)).abs()
    energy = (deviation * pure_mask).sum(dim=(1, 2)) / (pure_mask.sum(dim=(1, 2)) + 1e-8)
    return energy.cpu().numpy()

E_bg_real = pure_background_deviation(X_real_full)
E_bg_fake = pure_background_deviation(X_fake_full)
print(f"  Pearson(residual, gradcam) on REAL (n={N_CORR}): mean={r_real.mean():+.3f} ± {r_real.std():.3f}")
print(f"  Pearson(residual, gradcam) on FAKE (n={N_CORR}): mean={r_fake.mean():+.3f} ± {r_fake.std():.3f}")
print(f"  Pure-background deviation from -1: REAL={E_bg_real.mean():.5f}  "
      f"FAKE={E_bg_fake.mean():.5f}  "
      f"ratio FAKE/REAL = {E_bg_fake.mean() / max(E_bg_real.mean(), 1e-8):.1f}x")


def overlay_img(img, cam, alpha=0.5):
    img01 = (img + 1) / 2
    img_rgb = np.stack([img01] * 3, axis=-1)
    cmap = plt.get_cmap("jet")
    cam_rgb = cmap(cam)[:, :, :3]
    return (1 - alpha) * img_rgb + alpha * cam_rgb


# Layout: cột 0 là label, cột cuối là colorbar, ở giữa là N_VIS ô ảnh.
fig = plt.figure(figsize=(2.0 * N_VIS + 3.4, 9.5))
gs = fig.add_gridspec(6, N_VIS + 2,
                      width_ratios=[0.6] + [1.0] * N_VIS + [0.08],
                      hspace=0.18, wspace=0.06)

row_titles = [
    "REAL\nanh goc",
    "REAL\nhigh-freq\n|I − blur(I)|",
    "REAL\nGrad-CAM\noverlay",
    "FAKE (cGAN)\nanh goc",
    "FAKE (cGAN)\nhigh-freq\n|I − blur(I)|",
    "FAKE (cGAN)\nGrad-CAM\noverlay",
]

for j in range(N_VIS):
    img = X_real[j, 0].cpu().numpy()
    ax = fig.add_subplot(gs[0, j + 1])
    ax.imshow(img, cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"P(fake)={prob_real[j,1]:.2f}", fontsize=9)
    ax.axis("off")

    ax = fig.add_subplot(gs[1, j + 1])
    im_res_r = ax.imshow(res_real[j], cmap="jet", vmin=0, vmax=1)
    ax.axis("off")

    ax = fig.add_subplot(gs[2, j + 1])
    ax.imshow(overlay_img(img, cam_real[j]))
    ax.axis("off")

for j in range(N_VIS):
    img = X_fake[j, 0].cpu().numpy()
    ax = fig.add_subplot(gs[3, j + 1])
    ax.imshow(img, cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"P(fake)={prob_fake[j,1]:.2f}", fontsize=9)
    ax.axis("off")

    ax = fig.add_subplot(gs[4, j + 1])
    im_res_f = ax.imshow(res_fake[j], cmap="jet", vmin=0, vmax=1)
    ax.axis("off")

    ax = fig.add_subplot(gs[5, j + 1])
    ax.imshow(overlay_img(img, cam_fake[j]))
    ax.axis("off")

for r, txt in enumerate(row_titles):
    ax = fig.add_subplot(gs[r, 0])
    ax.text(0.95, 0.5, txt, transform=ax.transAxes,
            ha="right", va="center", fontsize=9.5, fontweight="bold")
    ax.axis("off")

cax = fig.add_subplot(gs[1:3, -1])
cbar = fig.colorbar(im_res_r, cax=cax)
cbar.set_label("0 = thap     →     1 = cao\n(high-freq / attention)",
               rotation=270, labelpad=22, fontsize=9)
cax2 = fig.add_subplot(gs[4:6, -1])
fig.colorbar(im_res_f, cax=cax2)

ratio_bg = E_bg_fake.mean() / max(E_bg_real.mean(), 1e-8)
fig.suptitle(
    "Grad-CAM = vung CNN nhin vao de quyet dinh 'fake' (do = nhin nhieu, xanh = bo qua)\n"
    f"Lech khoi -1 trong vung nen den thuan: REAL = {E_bg_real.mean():.5f} "
    f"(bang 0 — MNIST nen sach), FAKE = {E_bg_fake.mean():.5f} "
    f"→ fake bi jitter manh hon ~{ratio_bg:.0f}×.  "
    f"Pearson(jitter, Grad-CAM) = +{r_fake.mean():.2f}  "
    "→ CNN bam dung vao chinh vung jitter nay.",
    fontsize=10, y=0.998,
)

fig.savefig(f"{OUT_DIR}/gradcam_overlay.png", dpi=140, bbox_inches="tight")
plt.close()
print(f"Saved {OUT_DIR}/gradcam_overlay.png")


fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.5),
                         gridspec_kw={"width_ratios": [1, 1, 0.05]})
im0 = axes[0].imshow(cam_real_full.mean(0), cmap="jet", vmin=0, vmax=1)
axes[0].set_title(f"Mean Grad-CAM tren {N_CORR} REAL\n(huong 'fake')", fontsize=10)
axes[0].axis("off")
axes[1].imshow(cam_fake_full.mean(0), cmap="jet", vmin=0, vmax=1)
axes[1].set_title(f"Mean Grad-CAM tren {N_CORR} FAKE\n(huong 'fake')", fontsize=10)
axes[1].axis("off")
fig.colorbar(im0, cax=axes[2]).set_label("attention\n(0=thap, 1=cao)",
                                          rotation=270, labelpad=20, fontsize=9)
fig.suptitle("Trung binh: CNN luon chu y vao than chu, fake nong hon real",
             fontsize=11)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/gradcam_mean.png", dpi=140, bbox_inches="tight")
plt.close()
print(f"Saved {OUT_DIR}/gradcam_mean.png")


fig, ax = plt.subplots(figsize=(6.5, 4.5))
for j in range(N_VIS):
    ax.scatter(res_fake[j].ravel(), cam_fake[j].ravel(),
               s=4, alpha=0.25, label=f"fake #{j}" if j < 1 else None,
               color="C3")
for j in range(N_VIS):
    ax.scatter(res_real[j].ravel(), cam_real[j].ravel(),
               s=4, alpha=0.25, label=f"real #{j}" if j < 1 else None,
               color="C0")
ax.set_xlabel("high-freq residual (per-pixel, normalized)")
ax.set_ylabel("Grad-CAM attention (per-pixel)")
ax.set_title(f"Pixel-level: pixel cang nhieu high-freq -> CNN cang chu y\n"
             f"Pearson r (full val n={N_CORR}/lop): real={r_real.mean():+.3f}, "
             f"fake={r_fake.mean():+.3f}",
             fontsize=10)
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/gradcam_corr.png", dpi=140, bbox_inches="tight")
plt.close()
print(f"Saved {OUT_DIR}/gradcam_corr.png")

with open(f"{OUT_DIR}/gradcam.log", "w", encoding="utf-8") as fh:
    fh.write(f"N per class: {N_CORR}\n")
    fh.write(f"Pearson(high-freq residual, Grad-CAM)  REAL: "
             f"mean={r_real.mean():+.4f}  std={r_real.std():.4f}\n")
    fh.write(f"Pearson(high-freq residual, Grad-CAM)  FAKE: "
             f"mean={r_fake.mean():+.4f}  std={r_fake.std():.4f}\n")
    fh.write(f"Background high-freq energy mean(|I - blur(I)| * (I < -0.5)):\n")
    fh.write(f"  REAL: {E_bg_real.mean():.5f}  std={E_bg_real.std():.5f}\n")
    fh.write(f"  FAKE: {E_bg_fake.mean():.5f}  std={E_bg_fake.std():.5f}\n")
    fh.write(f"  ratio FAKE/REAL: {E_bg_fake.mean() / max(E_bg_real.mean(), 1e-8):.2f}x\n")
print(f"Saved {OUT_DIR}/gradcam.log")
print("\nDone. Inspect output/gradcam_overlay.png, gradcam_mean.png, gradcam_corr.png.")


In [ ]:
%%writefile exp2_pgan_texcnn.py
# TexCNN train từ đầu để phát hiện PGAN-DTD fake.
# Mục đích: kiểm tra detector nhỏ có theo nổi modern GAN hay không.
import os, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import torchvision
from torchvision import transforms
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# pytorch_GAN_zoo gọi Adam với betas dạng list. PyTorch 2.x từ chối khi mix
# float với Tensor, nên ép kiểu về float trước khi forward sang Adam gốc.
_orig_adam_init = optim.Adam.__init__
def _patched_adam_init(self, params, *args, **kwargs):
    if 'betas' in kwargs and kwargs['betas'] is not None:
        b = kwargs['betas']
        kwargs['betas'] = (float(b[0]), float(b[1]))
    return _orig_adam_init(self, params, *args, **kwargs)
optim.Adam.__init__ = _patched_adam_init

OUT_DIR     = "output"
DATA_DIR    = "../data"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
N_PER_CLASS = 1500
BS_PGAN     = 16
BATCH       = 32
EPOCHS      = 8
LR          = 5e-4
SEED        = 42
VAL_RATIO   = 0.2
IMG_SIZE    = 128

os.makedirs(OUT_DIR, exist_ok=True)


def build_dataset(log_fn=print):
    log_fn("\n" + "=" * 60); log_fn("Build PGAN-DTD dataset"); log_fn("=" * 60)

    pgan = torch.hub.load('facebookresearch/pytorch_GAN_zoo:hub', 'PGAN',
                          model_name='DTD', pretrained=True,
                          useGPU=(DEVICE == 'cuda'))
    log_fn("  Loaded PGAN-DTD")

    fakes = []
    for i in range(0, N_PER_CLASS, BS_PGAN):
        n = min(BS_PGAN, N_PER_CLASS - i)
        noise, _ = pgan.buildNoiseData(n)
        with torch.no_grad():
            x = pgan.test(noise)
        fakes.append(x.cpu()); del x, noise; gc.collect()
    fakes = torch.cat(fakes)
    # Ép PGAN output về [-1, 1] cho khớp với DTD reals đã chuẩn hoá.
    fakes = fakes.clamp(-1, 1)
    log_fn(f"  Fakes: {fakes.shape}  range [{fakes.min():.2f}, {fakes.max():.2f}]")

    # Gộp cả ba split của DTD vì mỗi split có ~1880 ảnh, gộp lại mới đủ N_PER_CLASS.
    tf = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
    ])
    reals_list = []
    for split in ['train', 'val', 'test']:
        ds = torchvision.datasets.DTD(DATA_DIR, split=split, download=True, transform=tf)
        loader = DataLoader(ds, batch_size=128, shuffle=True, num_workers=0)
        for batch, _ in loader:
            reals_list.append(batch)
            if sum(b.size(0) for b in reals_list) >= N_PER_CLASS:
                break
        if sum(b.size(0) for b in reals_list) >= N_PER_CLASS:
            break
    reals = torch.cat(reals_list)[:N_PER_CLASS]
    log_fn(f"  Reals: {reals.shape}  range [{reals.min():.2f}, {reals.max():.2f}]")

    X = torch.cat([reals, fakes], dim=0)
    y = torch.cat([torch.zeros(N_PER_CLASS, dtype=torch.long),
                   torch.ones (N_PER_CLASS, dtype=torch.long)])
    log_fn(f"  Combined: X={X.shape}  y={y.shape}  (label: 0=real, 1=fake)")
    return X, y


class TexCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,  16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(64 * 8 * 8, 128)
        self.fc2   = nn.Linear(128, 2)
        self.drop  = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.pool(F.relu(self.conv4(x)))
        x = x.flatten(1)
        x = self.drop(F.relu(self.fc1(x)))
        return self.fc2(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_one_epoch(model, loader, opt, loss_fn):
    model.train()
    total_loss = 0.0; correct = 0; n = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        logits = model(X)
        loss = loss_fn(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item() * X.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += X.size(0)
    return total_loss / n, correct / n


def evaluate(model, loader, loss_fn):
    model.train(False)
    total_loss = 0.0; correct = 0; n = 0
    all_pred = []; all_true = []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            logits = model(X)
            loss = loss_fn(logits, y)
            total_loss += loss.item() * X.size(0)
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            n += X.size(0)
            all_pred.append(pred.cpu()); all_true.append(y.cpu())
    return (total_loss / n, correct / n,
            torch.cat(all_pred).numpy(), torch.cat(all_true).numpy())


def confusion_matrix(y_true, y_pred):
    cm = np.zeros((2, 2), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def plot_confusion(cm, fname, title):
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["real (pred)", "fake (pred)"])
    ax.set_yticklabels(["real (true)", "fake (true)"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black",
                    fontsize=14, fontweight="bold")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/{fname}", dpi=130); plt.close()


def main():
    LOG = open(f"{OUT_DIR}/results_pgan.txt", "w", encoding="utf-8")
    def log(m=""):
        print(m); LOG.write(m + "\n"); LOG.flush()

    torch.manual_seed(SEED); np.random.seed(SEED)
    log(f"Device: {DEVICE}")
    log(f"N_PER_CLASS={N_PER_CLASS}  EPOCHS={EPOCHS}  BATCH={BATCH}  LR={LR}  IMG_SIZE={IMG_SIZE}")

    X, y = build_dataset(log)
    ds = TensorDataset(X, y)
    n_val = int(len(ds) * VAL_RATIO)
    n_train = len(ds) - n_val
    train_ds, val_ds = random_split(ds, [n_train, n_val],
                                    generator=torch.Generator().manual_seed(SEED))
    log(f"  Train/Val split: {n_train}/{n_val}")

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False)

    log("\n" + "=" * 60); log("Build TexCNN"); log("=" * 60)
    model = TexCNN().to(DEVICE)
    log(f"  Parameters: {count_params(model):,}")
    log(f"  Architecture (input 3x128x128):")
    log(f"    conv1+pool: 3->16,   128x128 -> 64x64")
    log(f"    conv2+pool: 16->32,   64x64 -> 32x32")
    log(f"    conv3+pool: 32->64,   32x32 -> 16x16")
    log(f"    conv4+pool: 64->64,   16x16 ->  8x8")
    log(f"    fc1: 64*8*8=4096 -> 128 + dropout(0.3)")
    log(f"    fc2: 128 -> 2 logits")

    opt = optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()

    log("\n" + "=" * 60); log("Training"); log("=" * 60)
    log(f"  {'Epoch':>6} {'TrainLoss':>10} {'TrainAcc':>10} {'ValLoss':>10} {'ValAcc':>10}")
    best_val_acc = 0.0
    for epoch in range(1, EPOCHS + 1):
        tl, ta = train_one_epoch(model, train_loader, opt, loss_fn)
        vl, va, _, _ = evaluate(model, val_loader, loss_fn)
        log(f"  {epoch:>6d} {tl:>10.4f} {ta:>10.4f} {vl:>10.4f} {va:>10.4f}")
        if va > best_val_acc:
            best_val_acc = va
            torch.save(model.state_dict(), f"{OUT_DIR}/cnn_pgan_best.pth")
    log(f"\n  Best val acc: {best_val_acc:.4f}  (saved to {OUT_DIR}/cnn_pgan_best.pth)")

    log("\n" + "=" * 60); log("Final evaluation (best)"); log("=" * 60)
    model.load_state_dict(torch.load(f"{OUT_DIR}/cnn_pgan_best.pth",
                                      map_location=DEVICE, weights_only=True))
    vl, va, pred, true = evaluate(model, val_loader, loss_fn)
    log(f"  Val accuracy: {va:.4f}")
    cm = confusion_matrix(true, pred)
    log(f"\n  Confusion matrix:")
    log(f"                 pred=real  pred=fake")
    log(f"    true=real    {cm[0,0]:>9d}  {cm[0,1]:>9d}")
    log(f"    true=fake    {cm[1,0]:>9d}  {cm[1,1]:>9d}")
    log(f"\n    Real recall (TN/(TN+FP)): {cm[0,0]/(cm[0,0]+cm[0,1]):.4f}")
    log(f"    Fake recall (TP/(TP+FN)): {cm[1,1]/(cm[1,0]+cm[1,1]):.4f}")
    log(f"    False Positive rate:      {cm[0,1]/(cm[0,0]+cm[0,1]):.4f}")
    log(f"    False Negative rate:      {cm[1,0]/(cm[1,0]+cm[1,1]):.4f}")
    plot_confusion(cm, "confusion_matrix_pgan.png",
                   "PGAN-DTD: CNN confusion matrix (val)")

    torch.save({
        "X": X, "y": y,
        "val_indices": val_ds.indices,
    }, f"{OUT_DIR}/dataset_pgan.pt")
    log(f"  Saved dataset_pgan.pt")

    LOG.close()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile exp3_pgan_resnet.py
# ResNet18 transfer trên PGAN-DTD; cùng dataset với TexCNN scratch ở exp2 nhưng đổi
# detector, để tách hiệu ứng transfer learning ra khỏi hiệu ứng kiến trúc GAN.
import os, gc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# pytorch_GAN_zoo gọi Adam với betas dạng list. PyTorch 2.x từ chối khi mix
# float với Tensor, nên ép kiểu về float trước khi forward sang Adam gốc.
_orig_adam_init = optim.Adam.__init__
def _patched_adam_init(self, params, *args, **kwargs):
    if 'betas' in kwargs and kwargs['betas'] is not None:
        b = kwargs['betas']
        kwargs['betas'] = (float(b[0]), float(b[1]))
    return _orig_adam_init(self, params, *args, **kwargs)
optim.Adam.__init__ = _patched_adam_init

OUT_DIR     = "output"
DATA_DIR    = "../data"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
N_PER_CLASS = 2500
IMG_SIZE    = 128
BATCH       = 32
SEED        = 42
VAL_RATIO   = 0.2
BS_PGAN     = 16

EPOCHS_HEAD = 3
EPOCHS_FT   = 12
LR_HEAD     = 1e-3
LR_FT       = 1e-4

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)


def build_dataset(log):
    log("\n" + "=" * 60); log("Build PGAN-DTD dataset"); log("=" * 60)

    pgan = torch.hub.load('facebookresearch/pytorch_GAN_zoo:hub', 'PGAN',
                          model_name='DTD', pretrained=True,
                          useGPU=(DEVICE == 'cuda'))
    log("  Loaded PGAN-DTD")

    fakes = []
    for i in range(0, N_PER_CLASS, BS_PGAN):
        n = min(BS_PGAN, N_PER_CLASS - i)
        noise, _ = pgan.buildNoiseData(n)
        with torch.no_grad():
            x = pgan.test(noise)
        fakes.append(x.cpu()); del x, noise; gc.collect()
        if (i // BS_PGAN) % 20 == 0:
            log(f"    sampled {i + n}/{N_PER_CLASS}")
    fakes = torch.cat(fakes).clamp(-1, 1)
    log(f"  Fakes: {fakes.shape}  range [{fakes.min():.2f}, {fakes.max():.2f}]")

    tf = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
    ])
    reals_list = []
    for split in ['train', 'val', 'test']:
        ds = torchvision.datasets.DTD(DATA_DIR, split=split, download=True, transform=tf)
        loader = DataLoader(ds, batch_size=128, shuffle=True, num_workers=0)
        for batch, _ in loader:
            reals_list.append(batch)
            if sum(b.size(0) for b in reals_list) >= N_PER_CLASS:
                break
        if sum(b.size(0) for b in reals_list) >= N_PER_CLASS:
            break
    reals = torch.cat(reals_list)[:N_PER_CLASS]
    log(f"  Reals: {reals.shape}  range [{reals.min():.2f}, {reals.max():.2f}]")

    X = torch.cat([reals, fakes], dim=0)
    y = torch.cat([torch.zeros(N_PER_CLASS, dtype=torch.long),
                   torch.ones(N_PER_CLASS, dtype=torch.long)])
    log(f"  Combined: X={X.shape}  y={y.shape}  (0=real, 1=fake)")
    return X, y


def build_resnet18():
    m = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m


def freeze_backbone(model):
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith("fc.")


def unfreeze_for_finetune(model):
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith("fc.") or name.startswith("layer4.")


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def renormalize_for_imagenet(x):
    x01 = (x + 1) / 2
    return (x01 - IMAGENET_MEAN.to(x.device)) / IMAGENET_STD.to(x.device)


def train_one_epoch(model, loader, opt, loss_fn, augment=True):
    model.train()
    total_loss = 0.0; correct = 0; n = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        if augment and torch.rand(1).item() < 0.5:
            X = torch.flip(X, dims=[3])
        X = renormalize_for_imagenet(X)
        logits = model(X)
        loss = loss_fn(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item() * X.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += X.size(0)
    return total_loss / n, correct / n


def evaluate(model, loader, loss_fn):
    model.train(False)
    total_loss = 0.0; correct = 0; n = 0
    all_pred = []; all_true = []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            X = renormalize_for_imagenet(X)
            logits = model(X)
            loss = loss_fn(logits, y)
            total_loss += loss.item() * X.size(0)
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            n += X.size(0)
            all_pred.append(pred.cpu()); all_true.append(y.cpu())
    return (total_loss / n, correct / n,
            torch.cat(all_pred).numpy(), torch.cat(all_true).numpy())


def confusion_matrix(y_true, y_pred):
    cm = np.zeros((2, 2), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def plot_confusion(cm, fname, title):
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["real (pred)", "fake (pred)"])
    ax.set_yticklabels(["real (true)", "fake (true)"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black",
                    fontsize=14, fontweight="bold")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/{fname}", dpi=130); plt.close()


def main():
    LOG = open(f"{OUT_DIR}/results_pgan_resnet.txt", "w", encoding="utf-8")
    def log(m=""):
        print(m); LOG.write(m + "\n"); LOG.flush()

    torch.manual_seed(SEED); np.random.seed(SEED)
    log(f"Device: {DEVICE}")
    log(f"N_PER_CLASS={N_PER_CLASS}  IMG_SIZE={IMG_SIZE}  BATCH={BATCH}")
    log(f"Transfer learning: head {EPOCHS_HEAD}ep @ lr={LR_HEAD}, "
        f"finetune {EPOCHS_FT}ep @ lr={LR_FT}")

    X, y = build_dataset(log)
    ds = TensorDataset(X, y)
    n_val = int(len(ds) * VAL_RATIO)
    n_train = len(ds) - n_val
    train_ds, val_ds = random_split(ds, [n_train, n_val],
                                    generator=torch.Generator().manual_seed(SEED))
    log(f"  Train/Val split: {n_train}/{n_val}")
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False)

    log("\n" + "=" * 60); log("Build ResNet18 transfer"); log("=" * 60)
    model = build_resnet18().to(DEVICE)
    freeze_backbone(model)
    log(f"  Total params:     {sum(p.numel() for p in model.parameters()):,}")
    log(f"  Trainable (head): {count_trainable(model):,}")

    loss_fn = nn.CrossEntropyLoss()

    log("\n" + "=" * 60); log("Phase 1: train head only"); log("=" * 60)
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                     lr=LR_HEAD)
    log(f"  {'Epoch':>6} {'TrainLoss':>10} {'TrainAcc':>10} {'ValLoss':>10} {'ValAcc':>10}")
    best_val_acc = 0.0
    for epoch in range(1, EPOCHS_HEAD + 1):
        tl, ta = train_one_epoch(model, train_loader, opt, loss_fn)
        vl, va, _, _ = evaluate(model, val_loader, loss_fn)
        log(f"  {epoch:>6d} {tl:>10.4f} {ta:>10.4f} {vl:>10.4f} {va:>10.4f}")
        if va > best_val_acc:
            best_val_acc = va
            torch.save(model.state_dict(), f"{OUT_DIR}/cnn_pgan_resnet_best.pth")

    log("\n" + "=" * 60); log("Phase 2: unfreeze layer4 + finetune"); log("=" * 60)
    unfreeze_for_finetune(model)
    log(f"  Trainable now: {count_trainable(model):,}")
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                     lr=LR_FT)
    log(f"  {'Epoch':>6} {'TrainLoss':>10} {'TrainAcc':>10} {'ValLoss':>10} {'ValAcc':>10}")
    for epoch in range(1, EPOCHS_FT + 1):
        tl, ta = train_one_epoch(model, train_loader, opt, loss_fn)
        vl, va, _, _ = evaluate(model, val_loader, loss_fn)
        log(f"  {epoch:>6d} {tl:>10.4f} {ta:>10.4f} {vl:>10.4f} {va:>10.4f}")
        if va > best_val_acc:
            best_val_acc = va
            torch.save(model.state_dict(), f"{OUT_DIR}/cnn_pgan_resnet_best.pth")

    log(f"\n  Best val acc: {best_val_acc:.4f}  "
        f"(saved to {OUT_DIR}/cnn_pgan_resnet_best.pth)")

    log("\n" + "=" * 60); log("Final eval (best ckpt)"); log("=" * 60)
    model.load_state_dict(torch.load(f"{OUT_DIR}/cnn_pgan_resnet_best.pth",
                                      map_location=DEVICE, weights_only=True))
    vl, va, pred, true = evaluate(model, val_loader, loss_fn)
    log(f"  Val accuracy: {va:.4f}")
    cm = confusion_matrix(true, pred)
    log(f"\n  Confusion matrix:")
    log(f"                 pred=real  pred=fake")
    log(f"    true=real    {cm[0,0]:>9d}  {cm[0,1]:>9d}")
    log(f"    true=fake    {cm[1,0]:>9d}  {cm[1,1]:>9d}")
    log(f"\n    Real recall: {cm[0,0]/(cm[0,0]+cm[0,1]):.4f}")
    log(f"    Fake recall: {cm[1,1]/(cm[1,0]+cm[1,1]):.4f}")
    plot_confusion(cm, "confusion_matrix_pgan_resnet.png",
                   "PGAN-DTD vs DTD: ResNet18 transfer")

    torch.save({"X": X, "y": y, "val_indices": val_ds.indices},
               f"{OUT_DIR}/dataset_pgan_resnet.pt")
    log(f"  Saved {OUT_DIR}/dataset_pgan_resnet.pt")

    LOG.close()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile exp4_biggan_resnet.py
# ResNet18 transfer trên BigGAN-128 fakes vs Imagenette reals.
# Reals: Imagenette-160 (10 lớp ImageNet, ~94 MB).
import os, gc, sys, urllib.request, tarfile
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR     = "output"
DATA_DIR    = "../data"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
N_PER_CLASS = 2500
IMG_SIZE    = 128
BATCH       = 32
SEED        = 42
VAL_RATIO   = 0.2
TRUNC       = 0.4

EPOCHS_HEAD = 3
EPOCHS_FT   = 12
LR_HEAD     = 1e-3
LR_FT       = 1e-4

IMAGENETTE_URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"
IMAGENETTE_DIR = os.path.join(DATA_DIR, "imagenette2-160")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)


def download_imagenette(log):
    if os.path.exists(IMAGENETTE_DIR):
        log(f"  Imagenette already at {IMAGENETTE_DIR}")
        return
    tgz_path = os.path.join(DATA_DIR, "imagenette2-160.tgz")
    if not os.path.exists(tgz_path):
        log(f"  Downloading {IMAGENETTE_URL}")
        urllib.request.urlretrieve(IMAGENETTE_URL, tgz_path)
        log(f"  Saved to {tgz_path}")
    log("  Extracting...")
    with tarfile.open(tgz_path, "r:gz") as tar:
        tar.extractall(DATA_DIR)
    log(f"  Done extracting to {IMAGENETTE_DIR}")


def load_imagenette_reals(n, log):
    download_imagenette(log)
    paths = []
    for split in ["train", "val"]:
        split_dir = os.path.join(IMAGENETTE_DIR, split)
        if not os.path.isdir(split_dir):
            continue
        for cls in sorted(os.listdir(split_dir)):
            cls_dir = os.path.join(split_dir, cls)
            # Bỏ qua các entry hidden / metadata như .DS_Store, Thumbs.db.
            if not os.path.isdir(cls_dir) or cls.startswith('.'):
                continue
            for f in os.listdir(cls_dir):
                if f.startswith('.'):
                    continue
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    paths.append(os.path.join(cls_dir, f))
    rng = np.random.RandomState(SEED)
    rng.shuffle(paths)
    paths = paths[:n]
    log(f"  Loading {len(paths)} Imagenette images")

    tf = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
    ])
    tensors = []
    for p in paths:
        img = Image.open(p).convert("RGB")
        tensors.append(tf(img))
    return torch.stack(tensors)


def sample_biggan_fakes(n, log):
    try:
        from pytorch_pretrained_biggan import (BigGAN, one_hot_from_int,
                                                truncated_noise_sample)
    except ImportError:
        log("  ERROR: cai dat 'pytorch-pretrained-biggan' truoc.")
        log("    pip install pytorch-pretrained-biggan")
        sys.exit(1)

    log(f"  Loading BigGAN-deep-128 (download ~340 MB lan dau)")
    bg = BigGAN.from_pretrained('biggan-deep-128').to(DEVICE)
    bg.train(False)

    BS = 16
    fakes = []
    rng = np.random.RandomState(SEED)
    for i in range(0, n, BS):
        b = min(BS, n - i)
        cls_ids = rng.randint(0, 1000, size=b).tolist()
        noise = truncated_noise_sample(truncation=TRUNC, batch_size=b, seed=SEED + i)
        cls_vec = one_hot_from_int(cls_ids, batch_size=b)
        noise_t = torch.from_numpy(noise).to(DEVICE)
        cls_t = torch.from_numpy(cls_vec).to(DEVICE)
        with torch.no_grad():
            out = bg(noise_t, cls_t, TRUNC)
        fakes.append(out.detach().cpu().clamp(-1, 1))
        del out, noise_t, cls_t; gc.collect()
        if (i // BS) % 10 == 0:
            log(f"    sampled {i + b}/{n}")
    return torch.cat(fakes)[:n]


def build_dataset(log):
    log("\n" + "=" * 60); log("Build BigGAN-Imagenette dataset"); log("=" * 60)

    fakes = sample_biggan_fakes(N_PER_CLASS, log)
    log(f"  Fakes:   {fakes.shape}  range [{fakes.min():.2f}, {fakes.max():.2f}]")
    reals = load_imagenette_reals(N_PER_CLASS, log)
    log(f"  Reals:   {reals.shape}  range [{reals.min():.2f}, {reals.max():.2f}]")

    X = torch.cat([reals, fakes], dim=0)
    y = torch.cat([torch.zeros(N_PER_CLASS, dtype=torch.long),
                   torch.ones(N_PER_CLASS, dtype=torch.long)])
    log(f"  Combined: X={X.shape}  y={y.shape}  (0=real, 1=fake)")
    return X, y


def build_resnet18():
    m = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m


def freeze_backbone(model):
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith("fc.")


def unfreeze_for_finetune(model):
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith("fc.") or name.startswith("layer4.")


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ResNet18 pretrained yêu cầu chuẩn hoá theo ImageNet stats; data đang ở
# [-1, 1] vì Normalize([0.5]*3, [0.5]*3), nên cần đưa lại về [0, 1] rồi áp
# mean/std của ImageNet ngay trong forward.
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def renormalize_for_imagenet(x):
    x01 = (x + 1) / 2
    return (x01 - IMAGENET_MEAN.to(x.device)) / IMAGENET_STD.to(x.device)


def train_one_epoch(model, loader, opt, loss_fn, augment=True):
    model.train()
    total_loss = 0.0; correct = 0; n = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        if augment and torch.rand(1).item() < 0.5:
            X = torch.flip(X, dims=[3])
        X = renormalize_for_imagenet(X)
        logits = model(X)
        loss = loss_fn(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item() * X.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += X.size(0)
    return total_loss / n, correct / n


def evaluate(model, loader, loss_fn):
    model.train(False)
    total_loss = 0.0; correct = 0; n = 0
    all_pred = []; all_true = []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            X = renormalize_for_imagenet(X)
            logits = model(X)
            loss = loss_fn(logits, y)
            total_loss += loss.item() * X.size(0)
            pred = logits.argmax(1)
            correct += (pred == y).sum().item()
            n += X.size(0)
            all_pred.append(pred.cpu()); all_true.append(y.cpu())
    return (total_loss / n, correct / n,
            torch.cat(all_pred).numpy(), torch.cat(all_true).numpy())


def confusion_matrix(y_true, y_pred):
    cm = np.zeros((2, 2), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def plot_confusion(cm, fname, title):
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["real (pred)", "fake (pred)"])
    ax.set_yticklabels(["real (true)", "fake (true)"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black",
                    fontsize=14, fontweight="bold")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/{fname}", dpi=130); plt.close()


def main():
    LOG = open(f"{OUT_DIR}/results_biggan.txt", "w", encoding="utf-8")
    def log(m=""):
        print(m); LOG.write(m + "\n"); LOG.flush()

    torch.manual_seed(SEED); np.random.seed(SEED)
    log(f"Device: {DEVICE}")
    log(f"N_PER_CLASS={N_PER_CLASS}  IMG_SIZE={IMG_SIZE}  BATCH={BATCH}")
    log(f"Transfer learning: head {EPOCHS_HEAD}ep @ lr={LR_HEAD}, "
        f"finetune {EPOCHS_FT}ep @ lr={LR_FT}")

    X, y = build_dataset(log)
    ds = TensorDataset(X, y)
    n_val = int(len(ds) * VAL_RATIO)
    n_train = len(ds) - n_val
    train_ds, val_ds = random_split(ds, [n_train, n_val],
                                    generator=torch.Generator().manual_seed(SEED))
    log(f"  Train/Val split: {n_train}/{n_val}")
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False)

    log("\n" + "=" * 60); log("Build ResNet18 transfer"); log("=" * 60)
    model = build_resnet18().to(DEVICE)
    freeze_backbone(model)
    log(f"  Total params:     {sum(p.numel() for p in model.parameters()):,}")
    log(f"  Trainable (head): {count_trainable(model):,}")

    loss_fn = nn.CrossEntropyLoss()

    log("\n" + "=" * 60); log("Phase 1: train head only"); log("=" * 60)
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                     lr=LR_HEAD)
    log(f"  {'Epoch':>6} {'TrainLoss':>10} {'TrainAcc':>10} {'ValLoss':>10} {'ValAcc':>10}")
    best_val_acc = 0.0
    for epoch in range(1, EPOCHS_HEAD + 1):
        tl, ta = train_one_epoch(model, train_loader, opt, loss_fn)
        vl, va, _, _ = evaluate(model, val_loader, loss_fn)
        log(f"  {epoch:>6d} {tl:>10.4f} {ta:>10.4f} {vl:>10.4f} {va:>10.4f}")
        if va > best_val_acc:
            best_val_acc = va
            torch.save(model.state_dict(), f"{OUT_DIR}/cnn_biggan_resnet_best.pth")

    log("\n" + "=" * 60); log("Phase 2: unfreeze layer4 + finetune"); log("=" * 60)
    unfreeze_for_finetune(model)
    log(f"  Trainable now: {count_trainable(model):,}")
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                     lr=LR_FT)
    log(f"  {'Epoch':>6} {'TrainLoss':>10} {'TrainAcc':>10} {'ValLoss':>10} {'ValAcc':>10}")
    for epoch in range(1, EPOCHS_FT + 1):
        tl, ta = train_one_epoch(model, train_loader, opt, loss_fn)
        vl, va, _, _ = evaluate(model, val_loader, loss_fn)
        log(f"  {epoch:>6d} {tl:>10.4f} {ta:>10.4f} {vl:>10.4f} {va:>10.4f}")
        if va > best_val_acc:
            best_val_acc = va
            torch.save(model.state_dict(), f"{OUT_DIR}/cnn_biggan_resnet_best.pth")

    log(f"\n  Best val acc: {best_val_acc:.4f}  "
        f"(saved to {OUT_DIR}/cnn_biggan_resnet_best.pth)")

    log("\n" + "=" * 60); log("Final eval (best ckpt)"); log("=" * 60)
    model.load_state_dict(torch.load(f"{OUT_DIR}/cnn_biggan_resnet_best.pth",
                                      map_location=DEVICE, weights_only=True))
    vl, va, pred, true = evaluate(model, val_loader, loss_fn)
    log(f"  Val accuracy: {va:.4f}")
    cm = confusion_matrix(true, pred)
    log(f"\n  Confusion matrix:")
    log(f"                 pred=real  pred=fake")
    log(f"    true=real    {cm[0,0]:>9d}  {cm[0,1]:>9d}")
    log(f"    true=fake    {cm[1,0]:>9d}  {cm[1,1]:>9d}")
    log(f"\n    Real recall: {cm[0,0]/(cm[0,0]+cm[0,1]):.4f}")
    log(f"    Fake recall: {cm[1,1]/(cm[1,0]+cm[1,1]):.4f}")
    plot_confusion(cm, "confusion_matrix_biggan.png",
                   "BigGAN-128 vs Imagenette: ResNet18 transfer")

    log("\nSaving sample grid...")
    sample_grid(X, y, log)

    torch.save({"X": X, "y": y, "val_indices": val_ds.indices},
               f"{OUT_DIR}/dataset_biggan.pt")
    log(f"  Saved {OUT_DIR}/dataset_biggan.pt")

    LOG.close()


def sample_grid(X, y, log, n=4):
    real_idx = np.where(y.numpy() == 0)[0][:n]
    fake_idx = np.where(y.numpy() == 1)[0][:n]
    fig, axes = plt.subplots(2, n, figsize=(2.2 * n, 4.6))
    for j, i in enumerate(real_idx):
        img = ((X[i].permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)
        axes[0, j].imshow(img); axes[0, j].axis("off")
    axes[0, 0].set_title("Imagenette real", loc="left", fontsize=11)
    for j, i in enumerate(fake_idx):
        img = ((X[i].permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)
        axes[1, j].imshow(img); axes[1, j].axis("off")
    axes[1, 0].set_title("BigGAN-128 fake", loc="left", fontsize=11)
    fig.suptitle(f"Sample: BigGAN-128 (truncation={TRUNC}) vs Imagenette real",
                 fontsize=11)
    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/biggan_samples.png", dpi=130, bbox_inches="tight")
    plt.close()
    log(f"  Saved {OUT_DIR}/biggan_samples.png")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile gradcam_resnet.py
# Grad-CAM cho ResNet18 transfer (BigGAN + PGAN). Target layer = model.layer4;
# với input 128x128, layer4 trả về feature map (B, 512, 4, 4), upsample lên 128x128.
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.models import ResNet18_Weights
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = "output"
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
N_VIS   = 4
SEED    = 7

torch.manual_seed(SEED); np.random.seed(SEED)


def build_resnet18():
    m = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m


IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def renormalize(x):
    x01 = (x + 1) / 2
    return (x01 - IMAGENET_MEAN.to(x.device)) / IMAGENET_STD.to(x.device)


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._fwd)
        target_layer.register_full_backward_hook(self._bwd)

    def _fwd(self, m, i, o):
        self.activations = o.detach()

    def _bwd(self, m, gi, go):
        self.gradients = go[0].detach()

    def __call__(self, x, class_idx):
        self.model.zero_grad()
        x_norm = renormalize(x)
        logits = self.model(x_norm)
        score = logits[:, class_idx].sum()
        score.backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=x.shape[2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze(1)
        for i in range(cam.size(0)):
            mn, mx = cam[i].min(), cam[i].max()
            if mx - mn > 1e-8:
                cam[i] = (cam[i] - mn) / (mx - mn)
            else:
                cam[i] = torch.zeros_like(cam[i])
        return cam.cpu().numpy()


def overlay_rgb(img, cam, alpha=0.5):
    img01 = (img + 1) / 2
    img_rgb = img01.permute(1, 2, 0).numpy().clip(0, 1)
    cmap = plt.get_cmap("jet")
    cam_rgb = cmap(cam)[:, :, :3]
    return ((1 - alpha) * img_rgb + alpha * cam_rgb).clip(0, 1)


def run_one(label, ckpt_path, dataset_path, out_png):
    print(f"\n[{label}] loading {ckpt_path}")
    if not os.path.exists(ckpt_path) or not os.path.exists(dataset_path):
        print(f"  Missing checkpoint or dataset, skipping {label}")
        return

    model = build_resnet18().to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE,
                                      weights_only=True))
    model.train(False)

    cache = torch.load(dataset_path, weights_only=True)
    X = cache["X"]; y = cache["y"]; val_idx = np.array(cache["val_indices"])
    val_y = y[val_idx].numpy()
    real_pool = val_idx[val_y == 0]
    fake_pool = val_idx[val_y == 1]
    rng = np.random.RandomState(SEED)
    real_sel = rng.choice(real_pool, size=N_VIS, replace=False)
    fake_sel = rng.choice(fake_pool, size=N_VIS, replace=False)

    X_real = X[real_sel].to(DEVICE)
    X_fake = X[fake_sel].to(DEVICE)

    gradcam = GradCAM(model, target_layer=model.layer4)
    cam_real = gradcam(X_real.clone().requires_grad_(True), class_idx=1)
    cam_fake = gradcam(X_fake.clone().requires_grad_(True), class_idx=1)

    with torch.no_grad():
        x_real_n = renormalize(X_real)
        x_fake_n = renormalize(X_fake)
        logits_real = model(x_real_n).cpu().numpy()
        logits_fake = model(x_fake_n).cpu().numpy()
    p_real = np.exp(logits_real) / np.exp(logits_real).sum(axis=1, keepdims=True)
    p_fake = np.exp(logits_fake) / np.exp(logits_fake).sum(axis=1, keepdims=True)

    fig = plt.figure(figsize=(2.0 * N_VIS + 2.5, 8.0))
    gs = fig.add_gridspec(4, N_VIS + 2,
                          width_ratios=[0.7] + [1.0] * N_VIS + [0.06],
                          hspace=0.18, wspace=0.06)
    row_titles = [
        f"REAL\nanh goc",
        f"REAL\nGrad-CAM\noverlay",
        f"FAKE ({label})\nanh goc",
        f"FAKE ({label})\nGrad-CAM\noverlay",
    ]

    last_im = None
    for j in range(N_VIS):
        img = X_real[j].cpu()
        ax = fig.add_subplot(gs[0, j + 1])
        img01 = ((img + 1) / 2).permute(1, 2, 0).numpy().clip(0, 1)
        ax.imshow(img01)
        ax.set_title(f"P(fake)={p_real[j,1]:.2f}", fontsize=9)
        ax.axis("off")

        ax = fig.add_subplot(gs[1, j + 1])
        ax.imshow(overlay_rgb(img, cam_real[j]))
        ax.axis("off")

        imgf = X_fake[j].cpu()
        ax = fig.add_subplot(gs[2, j + 1])
        imgf01 = ((imgf + 1) / 2).permute(1, 2, 0).numpy().clip(0, 1)
        ax.imshow(imgf01)
        ax.set_title(f"P(fake)={p_fake[j,1]:.2f}", fontsize=9)
        ax.axis("off")

        ax = fig.add_subplot(gs[3, j + 1])
        last_im = ax.imshow(overlay_rgb(imgf, cam_fake[j]))
        ax.axis("off")

    for r, txt in enumerate(row_titles):
        ax = fig.add_subplot(gs[r, 0])
        ax.text(0.95, 0.5, txt, transform=ax.transAxes,
                ha="right", va="center", fontsize=10, fontweight="bold")
        ax.axis("off")

    cax = fig.add_subplot(gs[1:4:2, -1])
    sm = plt.cm.ScalarMappable(cmap="jet", norm=plt.Normalize(0, 1))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label("attention\n(0=bo qua, 1=nhin)", rotation=270,
                   labelpad=24, fontsize=9)

    fig.suptitle(
        f"ResNet18 transfer Grad-CAM tren {label}: vung do = ResNet18 nhin "
        "vao de noi 'fake'\n"
        "ImageNet features expect natural-image statistics; do la nhung vung "
        "GAN sai lech distribution.",
        fontsize=10, y=0.995,
    )
    fig.savefig(out_png, dpi=140, bbox_inches="tight")
    plt.close()
    print(f"  Saved {out_png}")


def main():
    run_one("BigGAN-128",
            f"{OUT_DIR}/cnn_biggan_resnet_best.pth",
            f"{OUT_DIR}/dataset_biggan.pt",
            f"{OUT_DIR}/gradcam_biggan.png")
    run_one("PGAN-DTD",
            f"{OUT_DIR}/cnn_pgan_resnet_best.pth",
            f"{OUT_DIR}/dataset_pgan_resnet.pt",
            f"{OUT_DIR}/gradcam_pgan_resnet.png")
    print("\nDone.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile cross_test.py
# Cross-test: ResNet18 đã train trên BigGAN-Imagenette, đem test trực tiếp trên
# PGAN-DTD val. Không retrain. Mục đích: detector có tổng quát hoá cross-GAN không,
# hay bắt buộc phải train riêng cho từng cặp GAN/dataset.
import os, gc, sys
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
from torchvision import transforms, models

# pytorch_GAN_zoo gọi Adam với betas dạng list. PyTorch 2.x từ chối khi mix
# float với Tensor, nên ép kiểu về float trước khi forward sang Adam gốc.
_orig_adam_init = optim.Adam.__init__
def _patched_adam_init(self, params, *args, **kwargs):
    if 'betas' in kwargs and kwargs['betas'] is not None:
        b = kwargs['betas']
        kwargs['betas'] = (float(b[0]), float(b[1]))
    return _orig_adam_init(self, params, *args, **kwargs)
optim.Adam.__init__ = _patched_adam_init

OUT_DIR  = "output"
DATA_DIR = "../data"
# Ưu tiên checkpoint vừa train trong session hiện tại; rơi xuống bản đã commit
# trong colab_result/ nếu chưa chạy exp4.
CKPT_CANDIDATES = [
    "output/cnn_biggan_resnet_best.pth",
    "colab_result/cnn_biggan_resnet_best.pth",
]
N        = 500
IMG_SIZE = 128
BS_PGAN  = 16
BATCH    = 32
SEED     = 42


def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda"), "cuda"
    try:
        import torch_directml
        return torch_directml.device(), "directml"
    except ImportError:
        return torch.device("cpu"), "cpu"

DEVICE, DEV_NAME = pick_device()

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def renormalize_for_imagenet(x):
    x01 = (x + 1) / 2
    return (x01 - IMAGENET_MEAN.to(x.device)) / IMAGENET_STD.to(x.device)


def sample_pgan_fakes(n, log):
    log(f"  Loading PGAN-DTD pretrained")
    pgan = torch.hub.load('facebookresearch/pytorch_GAN_zoo:hub', 'PGAN',
                          model_name='DTD', pretrained=True, useGPU=False)
    fakes = []
    for i in range(0, n, BS_PGAN):
        b = min(BS_PGAN, n - i)
        noise, _ = pgan.buildNoiseData(b)
        with torch.no_grad():
            x = pgan.test(noise)
        fakes.append(x.cpu()); del x, noise; gc.collect()
        if (i // BS_PGAN) % 5 == 0:
            log(f"    sampled {i + b}/{n}")
    return torch.cat(fakes).clamp(-1, 1)


def load_dtd_reals(n, log):
    tf = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
    ])
    pool = []
    for split in ['train', 'val', 'test']:
        ds = torchvision.datasets.DTD(DATA_DIR, split=split, download=True, transform=tf)
        loader = DataLoader(ds, batch_size=128, shuffle=True, num_workers=0)
        for batch, _ in loader:
            pool.append(batch)
            if sum(b.size(0) for b in pool) >= n:
                break
        if sum(b.size(0) for b in pool) >= n:
            break
    log(f"  Loaded {n} DTD reals")
    return torch.cat(pool)[:n]


def build_resnet18_head():
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m


@torch.no_grad()
def predict_in_batches(model, X, batch=BATCH):
    out = []
    for i in range(0, len(X), batch):
        xb = X[i:i+batch].to(DEVICE)
        xb = renormalize_for_imagenet(xb)
        logits = model(xb)
        out.append(logits.argmax(1).cpu())
    return torch.cat(out)


def confusion_matrix(y_true, y_pred):
    cm = np.zeros((2, 2), dtype=np.int64)
    for t, p in zip(y_true.tolist(), y_pred.tolist()):
        cm[t, p] += 1
    return cm


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    LOG = open(f"{OUT_DIR}/results_cross.txt", "w", encoding="utf-8")
    def log(m=""):
        print(m); LOG.write(m + "\n"); LOG.flush()

    torch.manual_seed(SEED); np.random.seed(SEED)
    log(f"Device: {DEV_NAME}")
    log(f"Cross-test: ResNet18 da train tren BigGAN-Imagenette, ap len PGAN-DTD val")
    log(f"  N_per_class = {N}, IMG_SIZE = {IMG_SIZE}")
    log("")

    log("=" * 60); log("Build PGAN-DTD test data"); log("=" * 60)
    fakes = sample_pgan_fakes(N, log)
    log(f"  Fakes: {fakes.shape}  range [{fakes.min():.2f}, {fakes.max():.2f}]")
    reals = load_dtd_reals(N, log)
    log(f"  Reals: {reals.shape}  range [{reals.min():.2f}, {reals.max():.2f}]")

    X = torch.cat([reals, fakes], dim=0)
    y = torch.cat([torch.zeros(N, dtype=torch.long),
                   torch.ones (N, dtype=torch.long)])
    log(f"  Combined: X={tuple(X.shape)}  y={tuple(y.shape)}  (0=real DTD, 1=fake PGAN)")
    log("")

    log("=" * 60); log("Load detector ResNet18(BigGAN-Imagenette)"); log("=" * 60)
    model = build_resnet18_head()
    ckpt_path = next((p for p in CKPT_CANDIDATES if os.path.exists(p)), None)
    if ckpt_path is None:
        log(f"  ERROR: khong tim thay checkpoint trong {CKPT_CANDIDATES}")
        log(f"  Chay exp4_biggan_resnet.py truoc roi quay lai cross_test.")
        sys.exit(1)
    state = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    model.load_state_dict(state)
    model.train(False)
    model = model.to(DEVICE)
    log(f"  Loaded {ckpt_path}")
    log("")

    log("=" * 60); log("Inference"); log("=" * 60)
    pred = predict_in_batches(model, X)
    acc = (pred == y).float().mean().item()
    cm = confusion_matrix(y, pred)
    log(f"  Accuracy: {acc:.4f}")
    log("")
    log(f"  Confusion matrix (0=real DTD, 1=fake PGAN, predict 0/1):")
    log(f"                 pred=real  pred=fake")
    log(f"    true=real    {cm[0,0]:>9d}  {cm[0,1]:>9d}")
    log(f"    true=fake    {cm[1,0]:>9d}  {cm[1,1]:>9d}")
    log("")
    log(f"    Real recall: {cm[0,0]/(cm[0,0]+cm[0,1]):.4f}")
    log(f"    Fake recall: {cm[1,1]/(cm[1,0]+cm[1,1]):.4f}")
    LOG.close()


if __name__ == "__main__":
    main()


## 3. cGAN-MNIST + TinyCNN scratch

Train cGAN 30 epoch (~1 phút trên GPU) nếu chưa có checkpoint, sau đó TinyCNN 105k params train từ đầu trên 10k+10k. Tổng khoảng 1.5 phút.

In [ ]:
!python exp1_cgan_tinycnn.py

## 4. Grad-CAM TinyCNN

Grad-CAM trên `conv2`, kèm hệ số tương quan giữa high-freq residual và attention map. ~5 giây.

In [ ]:
!python gradcam_tinycnn.py

## 5. PGAN-DTD + TexCNN scratch (baseline)

Sample 1500 PGAN fakes + 1500 DTD reals. TexCNN 564k params train từ đầu, kết quả khoảng 62%. ~3 phút.

In [ ]:
!python exp2_pgan_texcnn.py

## 6. PGAN-DTD + ResNet18 transfer

ResNet18 pretrained ImageNet, transfer 2 phase: head 3 epoch rồi unfreeze layer4 finetune 12 epoch. Kỳ vọng khoảng 98%. ~5-6 phút.

In [ ]:
!python exp3_pgan_resnet.py

## 7. BigGAN-128 + Imagenette + ResNet18 transfer

Sample 2500 BigGAN fakes + 2500 Imagenette reals, cùng pipeline transfer. Lần đầu sẽ download ~340 MB BigGAN và ~94 MB Imagenette. ~6-7 phút.

In [ ]:
!python exp4_biggan_resnet.py

## 8. Grad-CAM ResNet18 (PGAN + BigGAN)

Inference 4 real + 4 fake mỗi GAN, target = `layer4`. ~30 giây.

In [ ]:
!python gradcam_resnet.py

## 9. Cross-test BigGAN sang PGAN

Lấy ResNet18 đã train trên BigGAN, test thẳng trên PGAN val mà không retrain, để xem detector có tổng quát hoá cross-GAN hay không.

In [ ]:
!python cross_test.py

## 10. Hiển thị kết quả inline

In [ ]:
# 1. cGAN-MNIST + TinyCNN scratch
from IPython.display import Image, display
import os
print('=' * 70)
print(' 1. cGAN-MNIST + TinyCNN scratch ')
print('=' * 70)
if os.path.exists('output/results.txt'):
    print(open('output/results.txt', encoding='utf-8').read())
else:
    print('(missing output/results.txt)')
for p in ["output/confusion_matrix.png", "output/gradcam_overlay.png", "output/gradcam_mean.png", "output/gradcam_corr.png"]:
    if os.path.exists(p):
        display(Image(p))
    else:
        print(f'(missing {p})')


In [ ]:
# 2. PGAN-DTD + TexCNN scratch (baseline)
from IPython.display import Image, display
import os
print('=' * 70)
print(' 2. PGAN-DTD + TexCNN scratch (baseline) ')
print('=' * 70)
if os.path.exists('output/results_pgan.txt'):
    print(open('output/results_pgan.txt', encoding='utf-8').read())
else:
    print('(missing output/results_pgan.txt)')
for p in ["output/confusion_matrix_pgan.png"]:
    if os.path.exists(p):
        display(Image(p))
    else:
        print(f'(missing {p})')


In [ ]:
# 3. PGAN-DTD + ResNet18 transfer
from IPython.display import Image, display
import os
print('=' * 70)
print(' 3. PGAN-DTD + ResNet18 transfer ')
print('=' * 70)
if os.path.exists('output/results_pgan_resnet.txt'):
    print(open('output/results_pgan_resnet.txt', encoding='utf-8').read())
else:
    print('(missing output/results_pgan_resnet.txt)')
for p in ["output/confusion_matrix_pgan_resnet.png", "output/gradcam_pgan_resnet.png"]:
    if os.path.exists(p):
        display(Image(p))
    else:
        print(f'(missing {p})')


In [ ]:
# 4. BigGAN-128 + Imagenette + ResNet18 transfer
from IPython.display import Image, display
import os
print('=' * 70)
print(' 4. BigGAN-128 + Imagenette + ResNet18 transfer ')
print('=' * 70)
if os.path.exists('output/results_biggan.txt'):
    print(open('output/results_biggan.txt', encoding='utf-8').read())
else:
    print('(missing output/results_biggan.txt)')
for p in ["output/biggan_samples.png", "output/confusion_matrix_biggan.png", "output/gradcam_biggan.png"]:
    if os.path.exists(p):
        display(Image(p))
    else:
        print(f'(missing {p})')


In [ ]:
# 5. Cross-test BigGAN sang PGAN
from IPython.display import Image, display
import os
print('=' * 70)
print(' 5. Cross-test BigGAN sang PGAN ')
print('=' * 70)
if os.path.exists('output/results_cross.txt'):
    print(open('output/results_cross.txt', encoding='utf-8').read())
else:
    print('(missing output/results_cross.txt)')
for p in []:
    if os.path.exists(p):
        display(Image(p))
    else:
        print(f'(missing {p})')


## 11. (Tuỳ chọn) Tải kết quả về máy

```python
from google.colab import files
import os
for f in [
    'output/results.txt', 'output/results_pgan.txt',
    'output/results_pgan_resnet.txt', 'output/results_biggan.txt',
    'output/results_cross.txt',
    'output/confusion_matrix.png', 'output/confusion_matrix_pgan.png',
    'output/confusion_matrix_pgan_resnet.png', 'output/confusion_matrix_biggan.png',
    'output/gradcam_overlay.png', 'output/gradcam_mean.png',
    'output/gradcam_corr.png', 'output/gradcam_pgan_resnet.png',
    'output/gradcam_biggan.png', 'output/biggan_samples.png',
    'output/cnn_best.pth', 'output/cnn_pgan_best.pth',
    'output/cnn_pgan_resnet_best.pth', 'output/cnn_biggan_resnet_best.pth',
]:
    if os.path.exists(f):
        files.download(f)
```
